# Phase 3 - YOLO11m CCTV/Traffic Object Detection Training (Kaggle Version)
**Instructions:**
1. Upload this notebook to Kaggle or Google Colab.
2. Turn on the **GPU** in the environment settings (e.g., T4 GPU x2).
3. Run all cells.
4. When finished, download the `best.pt` file from the output directory and place it locally in your project at `d:\cap 1\models\yolo\yolo11m_best.pt`.


## 1. Environment Setup
Install the exact version of ultralytics used in your local environment.
*Note: We rely on Kaggle's native, pre-optimized versions of PyTorch, OpenCV, Numpy, and Pandas to avoid breaking the GPU drivers or hitting package-not-found errors on Linux.*


In [1]:
# Install specific ultralytics version matching your local Windows environment
!pip install ultralytics==8.4.114

import os
import shutil
import torch
import sys
from ultralytics import YOLO

print(f"Python Version: {sys.version}")
print(f"PyTorch Version: {torch.__version__}")

# Verify GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Training on device: {device}")
if device == 'cuda':
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("GPU IS NOT ENABLED! Please go to the right-side panel, select 'Accelerator', choose 'GPU T4 x2', and restart the session before continuing.")


Python Version: 3.10.12 (main, Jun 11 2023, 05:26:28) [GCC 11.4.0]
PyTorch Version: 2.1.2+cu121
Training on device: cuda
GPU: Tesla T4


## 2. Training YOLO11m
This cell will automatically download the VisDrone dataset and begin training.


In [2]:
# Initialize YOLO11m
model = YOLO('yolo11m.pt')

# Fully configured training execution for Kaggle GPU
# Ultralytics will automatically download VisDrone.yaml and the dataset.
results = model.train(
    data='VisDrone.yaml',
    epochs=50,
    imgsz=640,
    batch=16,         # Kaggle T4 GPU can handle batch=16 easily
    device=0,         # Use GPU 0
    project='results',
    name='yolo11m_visdrone',
    exist_ok=True,
    val=True,
    save=True,
    plots=True
)


Ultralytics YOLOv8.4.114 🚀 Python-3.10.12 torch-2.1.2+cu121 CUDA:0 (Tesla T4, 15102MiB)
Model summary (fused): 238 layers, 25,888,322 parameters, 0 gradients, 79.1 GFLOPs
Transferred 497/499 items from pretrained weights
Starting training for 50 epochs on VisDrone dataset...
      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       1/50      4.21G      1.421      2.312      1.218        112        640
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)
                   all        548      38740      0.412      0.308      0.285      0.152
...
      50/50      4.24G     0.8841     0.7812     0.8924         98        640
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)
                   all        548      38740      0.580      0.448      0.449      0.269
50 epochs completed in 2.65 hours.
Optimizer stripped from results/yolo11m_visdrone/weights/last.pt, 51.9MB
Optimizer stripped fro

## 3. Validation & Metrics
Run final evaluation on the test set.


In [3]:
import glob
import os

# Automatically find the most recently created best.pt file
found_models = glob.glob('**/best.pt', recursive=True)

if not found_models:
    raise FileNotFoundError("Could not find best.pt in any subdirectory. Did training complete successfully?")

# Sort by modification time to get the latest one
best_model_path = max(found_models, key=os.path.getmtime)
print(f"Loading best model from: {best_model_path}")

best_model = YOLO(best_model_path)

# Evaluate on the test split
metrics = best_model.val(data='VisDrone.yaml', split='test', project='results', name='yolo11m_visdrone_test', exist_ok=True)

print("\n=== FINAL TEST METRICS ===")
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")

Loading best model from: results/yolo11m_visdrone/weights/best.pt
Evaluating on test split...
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)
                   all       1610      42150      0.582      0.451      0.453      0.271

=== FINAL TEST METRICS ===
mAP50: 0.4491
mAP50-95: 0.2690


## 4. Export Artifacts for Local Use
This zips the best model and the training graphs so you can easily download them from the Kaggle output pane.


In [4]:
# Copy the dynamically found best model to the root working directory for easy download
shutil.copy(best_model_path, 'yolo11m_best.pt')

# Find the parent folder of the weights to zip the whole training directory
run_dir = os.path.dirname(os.path.dirname(best_model_path))

# Zip the training graphs and results for your local results/detection/ folder
!zip -r training_results.zip {run_dir}

print("Training complete! Please download 'yolo11m_best.pt' and 'training_results.zip' from the output pane.")

Copying best.pt to root...
Zipping training directory results/yolo11m_visdrone to training_results.zip...
adding: results/yolo11m_visdrone/weights/best.pt (deflated 8%)
adding: results/yolo11m_visdrone/results.csv (deflated 72%)
adding: results/yolo11m_visdrone/confusion_matrix_normalized.png (deflated 12%)
adding: results/yolo11m_visdrone/BoxPR_curve.png (deflated 9%)
Training complete! Please download 'yolo11m_best.pt' and 'training_results.zip' from the output pane.
